In [1]:
# Ollama 기반 LlamaIndex PDF RAG 예제에 필요한 패키지와 모델 안내입니다.
# llamaIndex로 연결할 Ollama / PDF Reader 설치
# !pip install llama-index-llms-ollama
# !pip install llama-index-embeddings-ollama
# !pip install llama-index-readers-file
# !pip install pypdf
# !ollama pull gemma3n:e4b
# !ollama pull nomic-embed-text

In [2]:
# PDF 로더, Ollama LLM/임베딩, LlamaIndex 구성 요소를 불러옵니다.
import subprocess

from llama_index.readers.file import PDFReader
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader


In [3]:
# 로컬 Ollama 서버 주소와 사용할 LLM/임베딩 모델을 설정합니다.
ollama_base_url = 'http://localhost:11434'
ollama_llm_model = 'gemma3n:e4b'
ollama_embed_model = 'nomic-embed-text'

def installed_ollama_models() -> set[str]:
    try:
        result = subprocess.run(
            ['ollama', 'list'],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError as exc:
        raise RuntimeError('Ollama CLI가 설치되어 있지 않습니다. https://ollama.com 에서 설치하세요.') from exc
    except subprocess.CalledProcessError as exc:
        raise RuntimeError('Ollama 서버가 실행 중인지 확인하세요. 터미널에서 `ollama serve`를 실행하세요.') from exc

    names = set()
    for line in result.stdout.splitlines()[1:]:
        parts = line.split()
        if parts:
            names.add(parts[0])
    return names

def ensure_ollama_model(model_name: str) -> None:
    installed = installed_ollama_models()
    candidates = {model_name}
    if ':' not in model_name:
        candidates.add(f'{model_name}:latest')

    if installed.intersection(candidates):
        print(f'이미 설치됨: {model_name}')
        return

    print(f'Ollama 모델 다운로드 중: {model_name}')
    subprocess.run(['ollama', 'pull', model_name], check=True)

for model_name in [ollama_llm_model, ollama_embed_model]:
    ensure_ollama_model(model_name)

llm = Ollama(
    model=ollama_llm_model,
    base_url=ollama_base_url,
    request_timeout=120.0,
    temperature=0 # 낮을수록 좋은거. gpt 기준 0.7이 일반, 0.5가 plus, 0.2가 pro 정도 된다고 함
)
embed_model = OllamaEmbedding(
    model_name=ollama_embed_model,
    base_url=ollama_base_url,
    request_timeout=120.0
)

Settings.llm = llm
Settings.embed_model = embed_model

print('Ollama 설정 완료')



이미 설치됨: gemma3n:e4b
이미 설치됨: nomic-embed-text
Ollama 설정 완료


In [4]:
# 지정한 폴더의 PDF 파일을 Document 형태로 로드합니다.
documents = SimpleDirectoryReader(
    input_dir='../Data/pdf_sample1',
    file_extractor={'.pdf' : PDFReader()}
).load_data()

In [5]:
# 로드된 문서 개수를 확인합니다.
print(f'로드된 문서 수 : {len(documents)}')

로드된 문서 수 : 23


In [6]:
# 첫 번째 문서 내용을 출력해 로드 결과를 확인합니다.
print(documents[0])

Doc ID: 35e3cbc2-7944-4b98-b5c4-0c79b481630a
Text: 2024 미국의 인공지능(AI) 정책․전략 현황과 변화 방향     - AI 지배력 강화와 초강대국 유지를 위해
미국은 어떻게 변화하고 있는가?-


In [7]:
# 로드한 문서로 벡터 인덱스를 생성합니다.
index = VectorStoreIndex.from_documents(documents) # vector database 만든거

2026-06-02 11:43:11,532 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:43:13,332 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:43:14,335 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:43:15,811 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:43:15,872 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


In [8]:
# 벡터 인덱스를 질의 엔진으로 변환합니다.
query_engine = index.as_query_engine()

2026-06-02 11:43:15,939 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [9]:
# 질의 엔진에 질문을 보내 응답을 생성합니다.
response = query_engine.query('향후 AI는 어디까지 발전할거 같아? 한글로 대답해줘')

2026-06-02 11:43:16,028 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-06-02 11:43:39,490 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


In [10]:
# 생성된 최종 응답을 출력합니다.
print(response)

문서에 따르면, 미국은 AI 지배력 강화와 초강대국 유지를 위해 변화하고 있습니다. 또한 FTC는 AI 관련 문제에 대한 법적 권한을 충분히 가지고 있으며, 스푸핑 금지 규칙 개정안을 포함하여 새로운 규칙 제정을 착수했습니다. FTC가 문제시하는 주요 AI 관련 비즈니스 관행은 다음과 같습니다: 차별과 편견을 조장하는 알고리즘 이용, 과대광고, 딥페이크 사기, 불공정 경쟁, 프라이버시 침해 등입니다.


In [11]:
# 응답에 사용된 근거 노드의 metadata와 score를 확인합니다.
for node in response.source_nodes:
    print(node.score)

0.5978162074815649
0.5843048164452046
